In [1]:

# ============================== 0. CONFIGURATION ==============================

from pathlib import Path
import os, json, random, copy, warnings, time
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, log_loss, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon, binomtest

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

warnings.filterwarnings("ignore", category=UserWarning)

@dataclass
class CFG:
    DATASET_ROOT: str = "/kaggle/input/datasets/drtawfikrrahman/deep-learning-framework-for-human-dna"
    OUTPUT_DIR: str = "./dna_experiment_outputs"
    GROUP_CSV: str = ""                 # e.g. "/kaggle/input/meta/patient_labels.csv"
    GROUP_PATH_COLUMN: str = "filepath"
    GROUP_ID_COLUMN: str = "patient_id"
    GROUP_LABEL_COLUMN: str = "label"

    IMAGE_SIZE: int = 224
    BATCH_SIZE: int = 16
    EPOCHS: int = 30
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 1e-4
    NUM_WORKERS: int = 2
    SEED: int = 42
    EARLY_STOPPING_PATIENCE: int = 7
    DROPOUT: float = 0.35
    USE_CLASS_WEIGHTS: bool = True
    USE_AUGMENTATION: bool = True
    PRETRAINED: bool = True
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Set to False only if images are already known to be RGB.
    FORCE_RGB: bool = True
    MAX_GRADCAM_IMAGES_PER_CLASS: int = 3
    MC_DROPOUT_PASSES: int = 30
    BOOTSTRAP_REPETITIONS: int = 1000
    ROBUSTNESS_NOISE_STD: float = 0.05
    RUN_ABLATIONS: bool = True
    RUN_BASELINES: bool = True

cfg = CFG()
ROOT = Path(cfg.DATASET_ROOT)
OUT = Path(cfg.OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device(cfg.DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(cfg.SEED)
print("Device:", DEVICE)


# ============================== 1. DATA DISCOVERY ==============================

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

def discover_images(root):
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {root}\n"
            "Update CFG.DATASET_ROOT to the directory containing class subfolders."
        )
    records = []
    class_dirs = sorted([p for p in root.iterdir() if p.is_dir()])
    if not class_dirs:
        raise ValueError("No class subfolders found. Use one folder per class.")
    for class_dir in class_dirs:
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({"path": str(p), "label": class_dir.name})
    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("No supported images found.")
    return df

df = discover_images(ROOT)

# Optional metadata file permits patient-level splitting.
if cfg.GROUP_CSV:
    meta = pd.read_csv(cfg.GROUP_CSV)
    required = [cfg.GROUP_PATH_COLUMN, cfg.GROUP_ID_COLUMN, cfg.GROUP_LABEL_COLUMN]
    missing = [c for c in required if c not in meta.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")
    meta["_fullpath"] = meta[cfg.GROUP_PATH_COLUMN].astype(str).apply(
        lambda x: str((ROOT / x).resolve()) if not Path(x).is_absolute() else str(Path(x).resolve())
    )
    df["_fullpath"] = df["path"].apply(lambda x: str(Path(x).resolve()))
    df = df.merge(
        meta[["_fullpath", cfg.GROUP_ID_COLUMN, cfg.GROUP_LABEL_COLUMN]],
        on="_fullpath", how="inner"
    )
    df["label"] = df[cfg.GROUP_LABEL_COLUMN].astype(str)
    df["group"] = df[cfg.GROUP_ID_COLUMN].astype(str)
    df.drop(columns=["_fullpath"], inplace=True)
    if df.empty:
        raise ValueError("No images matched the metadata CSV. Check relative paths.")
else:
    df["group"] = df["path"]  # image-level split fallback; patient leakage cannot be ruled out.

# Remove unreadable images and exact duplicate files (hash-based).
import hashlib
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

valid_rows, seen_hashes, excluded = [], set(), []
for _, row in df.iterrows():
    try:
        with Image.open(row["path"]) as im:
            im.verify()
        digest = sha256_file(row["path"])
        if digest in seen_hashes:
            excluded.append({"path": row["path"], "reason": "exact_duplicate"})
            continue
        seen_hashes.add(digest)
        valid_rows.append(row.to_dict())
    except Exception as e:
        excluded.append({"path": row["path"], "reason": f"unreadable: {type(e).__name__}"})

df = pd.DataFrame(valid_rows).reset_index(drop=True)
pd.DataFrame(excluded).to_csv(OUT / "excluded_files.csv", index=False)
if df.empty:
    raise ValueError("No valid images remain after quality control.")

classes = sorted(df["label"].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["target"] = df["label"].map(class_to_idx).astype(int)
num_classes = len(classes)
if num_classes < 2:
    raise ValueError("At least two classes are required.")

print("\nDataset size:", len(df))
print("Classes:", classes)
print(df["label"].value_counts())


# ============================== 2. DATA CHARACTERISTICS ==============================

def inspect_image(path):
    with Image.open(path) as im:
        return im.size[0], im.size[1], im.mode

image_info = df["path"].apply(inspect_image)
df["width"] = [x[0] for x in image_info]
df["height"] = [x[1] for x in image_info]
df["mode"] = [x[2] for x in image_info]

summary = df.groupby("label").agg(
    image_count=("path", "count"),
    mean_width=("width", "mean"),
    mean_height=("height", "mean"),
    image_modes=("mode", lambda x: ", ".join(sorted(set(x))))
).reset_index()
summary["class_fraction"] = summary["image_count"] / len(df)
summary.to_csv(OUT / "dataset_characteristics.csv", index=False)

plt.figure(figsize=(9, 5))
plt.bar(summary["label"], summary["image_count"])
plt.title("Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(OUT / "01_dataset_class_distribution.png", dpi=300)
plt.close()

# Split 70/15/15. Group splitting is used if patient IDs are available.
if cfg.GROUP_CSV:
    # First hold out 15% test by group, then 15% validation from remaining groups.
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=cfg.SEED)
    trainval_idx, test_idx = next(gss1.split(df, df["target"], groups=df["group"]))
    trainval = df.iloc[trainval_idx].copy()
    test_df = df.iloc[test_idx].copy()
    relative_val = 0.15 / 0.85
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val, random_state=cfg.SEED)
    tr_idx, val_idx = next(gss2.split(
        trainval, trainval["target"], groups=trainval["group"]
    ))
    train_df = trainval.iloc[tr_idx].copy()
    val_df = trainval.iloc[val_idx].copy()
else:
    trainval, test_df = train_test_split(
        df, test_size=0.15, random_state=cfg.SEED, stratify=df["target"]
    )
    train_df, val_df = train_test_split(
        trainval, test_size=(0.15 / 0.85), random_state=cfg.SEED,
        stratify=trainval["target"]
    )

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    part.to_csv(OUT / f"{name}_split.csv", index=False)
    print(f"{name}: {len(part)} images | {part['label'].value_counts().to_dict()}")

# Verify no patient/group overlap when patient metadata is supplied.
if cfg.GROUP_CSV:
    assert set(train_df.group).isdisjoint(set(val_df.group))
    assert set(train_df.group).isdisjoint(set(test_df.group))
    assert set(val_df.group).isdisjoint(set(test_df.group))


# ============================== 3. PREPROCESSING AND AUGMENTATION ==============================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_aug = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
plain_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
])

class DNAImageDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["path"]) as im:
            im = im.convert("RGB") if cfg.FORCE_RGB else im.copy()
        if self.transform:
            im = self.transform(im)
        return im, int(row["target"]), row["path"]

def make_loader(frame, transform, shuffle=False, weighted=False):
    ds = DNAImageDataset(frame, transform)
    sampler = None
    if weighted and cfg.USE_CLASS_WEIGHTS:
        counts = frame["target"].value_counts().to_dict()
        weights = [1.0 / counts[int(y)] for y in frame["target"]]
        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        shuffle = False
    return DataLoader(
        ds, batch_size=cfg.BATCH_SIZE, shuffle=shuffle if sampler is None else False,
        sampler=sampler, num_workers=cfg.NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

train_loader = make_loader(train_df, train_aug if cfg.USE_AUGMENTATION else eval_transform,
                           shuffle=True, weighted=False)
train_eval_loader = make_loader(train_df, eval_transform)
val_loader = make_loader(val_df, eval_transform)
test_loader = make_loader(test_df, eval_transform)

# Save a visual grid of original and augmented examples.
def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for j in range(5):
    row = train_df.iloc[j % len(train_df)]
    with Image.open(row["path"]) as im:
        original = im.convert("RGB")
    axes[0, j].imshow(original)
    axes[0, j].set_title(f"Original\n{row['label']}", fontsize=8)
    aug_tensor = (train_aug if cfg.USE_AUGMENTATION else eval_transform)(original)
    axes[1, j].imshow(np.transpose(denormalize(aug_tensor).numpy(), (1, 2, 0)))
    axes[1, j].set_title("Preprocessed / augmented", fontsize=8)
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.tight_layout()
plt.savefig(OUT / "02_preprocessing_augmentation_examples.png", dpi=300)
plt.close()


# ============================== 4. MODEL DEFINITIONS ==============================

class SimpleCNN(nn.Module):
    def __init__(self, n_classes, dropout=0.35):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(256, n_classes)
    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.classifier(self.dropout(x))

def build_model(name="resnet18", pretrained=True, dropout=0.35):
    if name == "custom_cnn":
        return SimpleCNN(num_classes, dropout)
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, num_classes))
        return model
    if name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_features, num_classes))
        return model
    raise ValueError(f"Unknown model: {name}")

def get_class_weights(frame):
    y = frame["target"].values
    present = np.unique(y)
    w = compute_class_weight(class_weight="balanced", classes=present, y=y)
    result = np.ones(num_classes, dtype=np.float32)
    for c, value in zip(present, w):
        result[c] = value
    return torch.tensor(result, dtype=torch.float32, device=DEVICE)


# ============================== 5. TRAINING AND CONVERGENCE ==============================

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total, correct = 0.0, 0, 0
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(x)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)

def fit_model(model, train_dl, valid_dl, epochs=None, use_weights=True, run_name="model"):
    epochs = epochs or cfg.EPOCHS
    weights = get_class_weights(train_df) if use_weights else None
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_loss, best_state, wait = float("inf"), None, 0
    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(model, train_dl, criterion, optimizer)
        va_loss, va_acc = run_epoch(model, valid_dl, criterion)
        scheduler.step(va_loss)
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        print(f"{run_name} | epoch {epoch+1:02d}/{epochs} "
              f"train loss={tr_loss:.4f} acc={tr_acc:.3f} "
              f"val loss={va_loss:.4f} acc={va_acc:.3f}")
        if va_loss < best_loss:
            best_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= cfg.EARLY_STOPPING_PATIENCE:
                print("Early stopping.")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

def plot_history(history, name):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epochs, history["train_loss"], label="Training")
    ax[0].plot(epochs, history["val_loss"], label="Validation")
    ax[0].set_title("Loss convergence"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
    ax[0].legend()
    ax[1].plot(epochs, history["train_acc"], label="Training")
    ax[1].plot(epochs, history["val_acc"], label="Validation")
    ax[1].set_title("Accuracy convergence"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy")
    ax[1].legend()
    plt.tight_layout()
    plt.savefig(OUT / f"03_convergence_{name}.png", dpi=300)
    plt.close()

def predict(model, loader):
    model.eval()
    all_y, all_prob, all_paths = [], [], []
    with torch.no_grad():
        for x, y, paths in loader:
            logits = model(x.to(DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_y.extend(y.numpy().tolist())
            all_prob.extend(probs.tolist())
            all_paths.extend(list(paths))
    return np.array(all_y), np.array(all_prob), all_paths

# Main proposed model: pretrained ResNet-18 with augmentation, class-weighted loss.
main_model = build_model("resnet18", pretrained=cfg.PRETRAINED, dropout=cfg.DROPOUT).to(DEVICE)
main_model, main_history = fit_model(main_model, train_loader, val_loader,
                                     use_weights=cfg.USE_CLASS_WEIGHTS, run_name="proposed_resnet18")
plot_history(main_history, "proposed_resnet18")
torch.save(main_model.state_dict(), OUT / "proposed_resnet18_best.pth")

y_test, prob_test, test_paths = predict(main_model, test_loader)
pred_test = prob_test.argmax(axis=1)


# ============================== 6. CLASSIFICATION METRICS ==============================

def classification_metrics(y_true, y_pred, probabilities):
    p, r, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(num_classes), zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": np.mean(p),
        "macro_recall": np.mean(r),
        "macro_f1": np.mean(f1),
        "weighted_f1": precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )[2],
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "log_loss": log_loss(y_true, probabilities, labels=np.arange(num_classes)),
        "class_precision": p, "class_recall": r, "class_f1": f1, "support": support
    }

metrics = classification_metrics(y_test, pred_test, prob_test)
pd.DataFrame([{
    k: v for k, v in metrics.items()
    if np.isscalar(v)
}]).to_csv(OUT / "04_overall_classification_metrics.csv", index=False)

report = classification_report(
    y_test, pred_test, labels=np.arange(num_classes),
    target_names=classes, output_dict=True, zero_division=0
)
pd.DataFrame(report).transpose().to_csv(OUT / "05_classwise_classification_report.csv")

cm = confusion_matrix(y_test, pred_test, labels=np.arange(num_classes))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_xticks(range(num_classes)); ax.set_yticks(range(num_classes))
ax.set_xticklabels(classes, rotation=40, ha="right")
ax.set_yticklabels(classes)
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(OUT / "06_confusion_matrix.png", dpi=300)
plt.close()


# ============================== 7. ROC AND PRECISION-RECALL ==============================

y_bin = label_binarize(y_test, classes=np.arange(num_classes))
if num_classes == 2 and y_bin.shape[1] == 1:
    y_bin = np.column_stack([1 - y_bin[:, 0], y_bin[:, 0]])

plt.figure(figsize=(7, 6))
roc_rows = []
for i, cls in enumerate(classes):
    if len(np.unique(y_bin[:, i])) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], prob_test[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.3f})")
    roc_rows.append({"class": cls, "roc_auc": roc_auc})
plt.plot([0, 1], [0, 1], "--", label="Chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "07_roc_curves.png", dpi=300); plt.close()
pd.DataFrame(roc_rows).to_csv(OUT / "roc_auc_by_class.csv", index=False)

plt.figure(figsize=(7, 6))
pr_rows = []
for i, cls in enumerate(classes):
    if len(np.unique(y_bin[:, i])) < 2:
        continue
    precision, recall, _ = precision_recall_curve(y_bin[:, i], prob_test[:, i])
    ap = average_precision_score(y_bin[:, i], prob_test[:, i])
    plt.plot(recall, precision, label=f"{cls} (AP={ap:.3f})")
    pr_rows.append({"class": cls, "average_precision": ap})
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title("One-vs-Rest Precision–Recall Curves")
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(OUT / "08_precision_recall_curves.png", dpi=300); plt.close()
pd.DataFrame(pr_rows).to_csv(OUT / "average_precision_by_class.csv", index=False)


# ============================== 8. BASELINE / TRANSFER-LEARNING COMPARISON ==============================

comparison_rows = [{
    "model": "Proposed ResNet18",
    "accuracy": metrics["accuracy"],
    "balanced_accuracy": metrics["balanced_accuracy"],
    "macro_f1": metrics["macro_f1"],
    "weighted_f1": metrics["weighted_f1"]
}]

if cfg.RUN_BASELINES:
    baseline_specs = [
        ("Custom CNN", "custom_cnn", False),
        ("ResNet18 (scratch)", "resnet18", False),
        ("EfficientNet-B0", "efficientnet_b0", cfg.PRETRAINED),
    ]
    for display_name, model_name, pretrained in baseline_specs:
        model = build_model(model_name, pretrained=pretrained, dropout=cfg.DROPOUT).to(DEVICE)
        model, hist = fit_model(model, train_loader, val_loader,
                                use_weights=cfg.USE_CLASS_WEIGHTS, run_name=display_name)
        plot_history(hist, model_name + "_comparison")
        yt, pp, _ = predict(model, test_loader)
        mm = classification_metrics(yt, pp.argmax(1), pp)
        comparison_rows.append({
            "model": display_name, "accuracy": mm["accuracy"],
            "balanced_accuracy": mm["balanced_accuracy"],
            "macro_f1": mm["macro_f1"], "weighted_f1": mm["weighted_f1"]
        })
        torch.save(model.state_dict(), OUT / f"{model_name}_comparison.pth")
    pd.DataFrame(comparison_rows).to_csv(OUT / "09_model_comparison.csv", index=False)
    comp = pd.DataFrame(comparison_rows)
    plt.figure(figsize=(9, 5))
    x = np.arange(len(comp))
    width = 0.2
    for k, metric_name in enumerate(["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]):
        plt.bar(x + (k - 1.5) * width, comp[metric_name], width, label=metric_name)
    plt.xticks(x, comp["model"], rotation=25, ha="right")
    plt.ylim(0, 1.05); plt.ylabel("Score"); plt.title("Model Comparison")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUT / "10_model_comparison.png", dpi=300); plt.close()


# ============================== 9. ABLATION STUDY ==============================

if cfg.RUN_ABLATIONS:
    ablation_rows = []
    # Controlled variants: same split, model, epochs and optimizer.
    variants = [
        ("Full proposed", True, True, cfg.DROPOUT),
        ("No augmentation", False, True, cfg.DROPOUT),
        ("No class weighting", True, False, cfg.DROPOUT),
        ("No dropout", True, True, 0.0),
    ]
    for name, aug_on, weights_on, dropout in variants:
        if name == "Full proposed":
            yt, pp = y_test, prob_test
        else:
            tr_dl = make_loader(train_df, train_aug if aug_on else eval_transform,
                                shuffle=True)
            model = build_model("resnet18", pretrained=cfg.PRETRAINED, dropout=dropout).to(DEVICE)
            model, _ = fit_model(model, tr_dl, val_loader, use_weights=weights_on,
                                 run_name="ablation_" + name.replace(" ", "_"))
            yt, pp, _ = predict(model, test_loader)
        mm = classification_metrics(yt, pp.argmax(1), pp)
        ablation_rows.append({
            "variant": name, "accuracy": mm["accuracy"],
            "balanced_accuracy": mm["balanced_accuracy"], "macro_f1": mm["macro_f1"]
        })
    pd.DataFrame(ablation_rows).to_csv(OUT / "11_ablation_study.csv", index=False)


# ============================== 10. GRAD-CAM EXPLAINABILITY ==============================

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.h1 = target_layer.register_forward_hook(self._forward_hook)
        self.h2 = target_layer.register_full_backward_hook(self._backward_hook)
    def _forward_hook(self, module, inp, out):
        self.activations = out
    def _backward_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]
    def generate(self, x, class_idx=None):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(logits.argmax(1).item())
        logits[:, class_idx].sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam
    def close(self):
        self.h1.remove(); self.h2.remove()

# ResNet18 final convolutional block.
cam_engine = GradCAM(main_model, main_model.layer4[-1])
cam_rows = []
for class_idx, cls in enumerate(classes):
    candidates = np.where(y_test == class_idx)[0][:cfg.MAX_GRADCAM_IMAGES_PER_CLASS]
    for image_idx in candidates:
        path = test_paths[image_idx]
        with Image.open(path) as im:
            rgb = im.convert("RGB")
            x = eval_transform(rgb).unsqueeze(0).to(DEVICE)
            cam = cam_engine.generate(x, int(pred_test[image_idx]))
            base = np.array(rgb.resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)))
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(base)
        ax.imshow(cam, cmap="jet", alpha=0.40)
        ax.set_title(
            f"True: {idx_to_class[int(y_test[image_idx])]} | "
            f"Pred: {idx_to_class[int(pred_test[image_idx])]}"
        )
        ax.axis("off"); plt.tight_layout()
        safe_cls = "".join(ch if ch.isalnum() else "_" for ch in cls)
        fname = f"gradcam_{safe_cls}_{image_idx}.png"
        plt.savefig(OUT / fname, dpi=250); plt.close()
        cam_rows.append({
            "file": path, "true_class": idx_to_class[int(y_test[image_idx])],
            "predicted_class": idx_to_class[int(pred_test[image_idx])],
            "confidence": float(prob_test[image_idx].max()), "heatmap": fname
        })
cam_engine.close()
pd.DataFrame(cam_rows).to_csv(OUT / "12_gradcam_index.csv", index=False)


# ============================== 11. CALIBRATION AND UNCERTAINTY ==============================

def expected_calibration_error(y_true, probs, n_bins=10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.any():
            conf = confidences[mask].mean()
            acc = accuracies[mask].mean()
            ece += mask.mean() * abs(acc - conf)
            rows.append({"bin_lower": lo, "bin_upper": hi, "count": int(mask.sum()),
                         "mean_confidence": conf, "accuracy": acc})
    return ece, pd.DataFrame(rows)

ece, calibration_bins = expected_calibration_error(y_test, prob_test)
calibration_bins.to_csv(OUT / "13_calibration_bins.csv", index=False)

plt.figure(figsize=(6, 6))
if len(calibration_bins):
    plt.plot(calibration_bins["mean_confidence"], calibration_bins["accuracy"],
             marker="o", label="Model")
plt.plot([0, 1], [0, 1], "--", label="Perfect calibration")
plt.xlabel("Mean predicted confidence"); plt.ylabel("Observed accuracy")
plt.title(f"Reliability Diagram (ECE={ece:.3f})")
plt.legend(); plt.tight_layout()
plt.savefig(OUT / "13_reliability_diagram.png", dpi=300); plt.close()

# MC dropout: activate dropout layers only, keeping batch normalization in eval mode.
def mc_dropout_predict(model, loader, passes=30):
    # Keep BatchNorm layers in evaluation mode, but activate Dropout layers.
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()
    all_passes = []
    for _ in range(passes):
        pass_probs = []
        with torch.no_grad():
            for x, _, _ in loader:
                logits = model(x.to(DEVICE))
                pass_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        all_passes.append(np.asarray(pass_probs))
    model.eval()
    stack = np.stack(all_passes, axis=0)
    mean_probs = stack.mean(axis=0)
    predictive_entropy = -(mean_probs * np.log(mean_probs + 1e-10)).sum(axis=1)
    expected_entropy = -(stack * np.log(stack + 1e-10)).sum(axis=2).mean(axis=0)
    mutual_information = predictive_entropy - expected_entropy
    return mean_probs, predictive_entropy, mutual_information

mc_probs, pred_entropy, mutual_info = mc_dropout_predict(
    main_model, test_loader, cfg.MC_DROPOUT_PASSES
)
uncertainty_df = pd.DataFrame({
    "filepath": test_paths, "true_class": [idx_to_class[int(x)] for x in y_test],
    "predicted_class": [idx_to_class[int(x)] for x in mc_probs.argmax(1)],
    "confidence": mc_probs.max(1), "predictive_entropy": pred_entropy,
    "mutual_information": mutual_info,
    "correct": (mc_probs.argmax(1) == y_test)
})
uncertainty_df.to_csv(OUT / "14_prediction_uncertainty.csv", index=False)


# ============================== 12. ROBUSTNESS AND ERROR ANALYSIS ==============================

class AddGaussianNoise:
    def __init__(self, std): self.std = std
    def __call__(self, tensor):
        return (tensor + torch.randn_like(tensor) * self.std).clamp(0, 1)

# Test clean images against mild pixel-space Gaussian noise.
noise_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    AddGaussianNoise(cfg.ROBUSTNESS_NOISE_STD),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
noise_loader = make_loader(test_df, noise_transform)
yn, pn, _ = predict(main_model, noise_loader)
clean_acc = accuracy_score(y_test, pred_test)
noise_acc = accuracy_score(yn, pn.argmax(1))
robustness = pd.DataFrame([
    {"condition": "Clean", "accuracy": clean_acc},
    {"condition": f"Gaussian noise std={cfg.ROBUSTNESS_NOISE_STD}", "accuracy": noise_acc}
])
robustness.to_csv(OUT / "15_robustness_results.csv", index=False)

errors = pd.DataFrame({
    "filepath": test_paths,
    "true_class": [idx_to_class[int(x)] for x in y_test],
    "predicted_class": [idx_to_class[int(x)] for x in pred_test],
    "confidence": prob_test.max(axis=1),
    "correct": pred_test == y_test
})
errors[~errors["correct"]].sort_values("confidence", ascending=False).to_csv(
    OUT / "16_misclassified_examples.csv", index=False
)

plt.figure(figsize=(7, 4))
plt.hist(errors.loc[errors.correct, "confidence"], bins=10, alpha=0.65, label="Correct")
plt.hist(errors.loc[~errors.correct, "confidence"], bins=10, alpha=0.65, label="Incorrect")
plt.xlabel("Maximum predicted probability"); plt.ylabel("Number of images")
plt.title("Confidence Distribution: Correct vs Incorrect")
plt.legend(); plt.tight_layout()
plt.savefig(OUT / "16_error_confidence_distribution.png", dpi=300); plt.close()


# ============================== 13. STATISTICAL SIGNIFICANCE / CONFIDENCE INTERVALS ==============================

def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        try:
            values.append(metric_fn(y_true[idx], y_pred[idx]))
        except Exception:
            pass
    if not values:
        return np.nan, np.nan
    return tuple(np.percentile(values, [2.5, 97.5]))

metric_functions = {
    "accuracy": accuracy_score,
    "balanced_accuracy": balanced_accuracy_score,
    "macro_f1": lambda a, b: precision_recall_fscore_support(
        a, b, average="macro", zero_division=0
    )[2]
}
ci_rows = []
for name, fn in metric_functions.items():
    lo, hi = bootstrap_metric_ci(
        y_test, pred_test, fn, cfg.BOOTSTRAP_REPETITIONS, cfg.SEED
    )
    ci_rows.append({"metric": name, "estimate": fn(y_test, pred_test),
                    "ci_95_lower": lo, "ci_95_upper": hi})
pd.DataFrame(ci_rows).to_csv(OUT / "17_bootstrap_confidence_intervals.csv", index=False)

# McNemar exact test between proposed model and the first available baseline.
# This requires paired predictions on the exact same test images.
if cfg.RUN_BASELINES and len(comparison_rows) > 1:
    # Re-load/recreate the custom CNN baseline to obtain paired predictions.
    baseline_path = OUT / "custom_cnn_comparison.pth"
    if baseline_path.exists():
        baseline_model = build_model("custom_cnn", pretrained=False).to(DEVICE)
        baseline_model.load_state_dict(torch.load(baseline_path, map_location=DEVICE))
        yb, pb, _ = predict(baseline_model, test_loader)
        base_pred = pb.argmax(1)
        proposed_correct = (pred_test == y_test)
        baseline_correct = (base_pred == y_test)
        n01 = int(np.sum(proposed_correct & ~baseline_correct))
        n10 = int(np.sum(~proposed_correct & baseline_correct))
        mcnemar_p = binomtest(min(n01, n10), n=n01+n10, p=0.5,
                              alternative="two-sided").pvalue if (n01+n10) else 1.0
        pd.DataFrame([{
            "comparison": "Proposed ResNet18 vs Custom CNN",
            "proposed_only_correct": n01,
            "baseline_only_correct": n10,
            "exact_mcnemar_p_value": mcnemar_p
        }]).to_csv(OUT / "18_mcnemar_test.csv", index=False)


# ============================== 14. SUMMARY OF EXPERIMENTAL FINDINGS ==============================

summary_metrics = {
    "dataset_total_images_after_qc": int(len(df)),
    "number_of_classes": int(num_classes),
    "class_names": classes,
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "split_strategy": "patient/group-level" if cfg.GROUP_CSV else "image-level",
    "patient_level_leakage_controlled": bool(cfg.GROUP_CSV),
    "test_accuracy": float(metrics["accuracy"]),
    "test_balanced_accuracy": float(metrics["balanced_accuracy"]),
    "test_macro_f1": float(metrics["macro_f1"]),
    "test_weighted_f1": float(metrics["weighted_f1"]),
    "test_mcc": float(metrics["mcc"]),
    "test_cohen_kappa": float(metrics["cohen_kappa"]),
    "test_log_loss": float(metrics["log_loss"]),
    "expected_calibration_error": float(ece),
    "clean_test_accuracy": float(clean_acc),
    "gaussian_noise_test_accuracy": float(noise_acc),
    "misclassified_test_images": int((~errors["correct"]).sum()),
    "important_interpretation": (
        "Preliminary image-classification performance only. These results do not "
        "establish clinical diagnostic validity or prospective early-detection utility."
    )
}
with open(OUT / "19_experimental_findings_summary.json", "w") as f:
    json.dump(summary_metrics, f, indent=2)

with open(OUT / "19_experimental_findings_summary.txt", "w") as f:
    f.write("SUMMARY OF EXPERIMENTAL FINDINGS\n")
    f.write("=" * 40 + "\n")
    for key, value in summary_metrics.items():
        f.write(f"{key}: {value}\n")

print("\nEXPERIMENT COMPLETE")
print(json.dumps(summary_metrics, indent=2))
print(f"\nAll tables, plots, model weights and reports saved in: {OUT.resolve()}")

Device: cuda

Dataset size: 10083
Classes: ['Chromosomekaryotype images', 'Fluorescent Neuronal Cells', 'Sequencing-derived visualization']
label
Chromosomekaryotype images          5474
Sequencing-derived visualization    4041
Fluorescent Neuronal Cells           568
Name: count, dtype: int64
train: 7057 images | {'Chromosomekaryotype images': 3831, 'Sequencing-derived visualization': 2828, 'Fluorescent Neuronal Cells': 398}
validation: 1513 images | {'Chromosomekaryotype images': 821, 'Sequencing-derived visualization': 607, 'Fluorescent Neuronal Cells': 85}
test: 1513 images | {'Chromosomekaryotype images': 822, 'Sequencing-derived visualization': 606, 'Fluorescent Neuronal Cells': 85}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 179MB/s]


proposed_resnet18 | epoch 01/30 train loss=0.0196 acc=0.995 val loss=0.0001 acc=1.000
proposed_resnet18 | epoch 02/30 train loss=0.0026 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 03/30 train loss=0.0005 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 04/30 train loss=0.0117 acc=0.999 val loss=0.0010 acc=1.000
proposed_resnet18 | epoch 05/30 train loss=0.0004 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 06/30 train loss=0.0003 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 07/30 train loss=0.0001 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 08/30 train loss=0.0117 acc=0.999 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 09/30 train loss=0.0004 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 10/30 train loss=0.0002 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 11/30 train loss=0.0011 acc=1.000 val loss=0.0000 acc=1.000
proposed_resnet18 | epoch 12/30 train loss=0.0012 acc=

100%|██████████| 20.5M/20.5M [00:00<00:00, 154MB/s]


EfficientNet-B0 | epoch 01/30 train loss=0.0771 acc=0.990 val loss=0.0006 acc=1.000
EfficientNet-B0 | epoch 02/30 train loss=0.0027 acc=1.000 val loss=0.0003 acc=1.000
EfficientNet-B0 | epoch 03/30 train loss=0.0027 acc=0.999 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 04/30 train loss=0.0007 acc=1.000 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 05/30 train loss=0.0012 acc=1.000 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 06/30 train loss=0.0005 acc=1.000 val loss=0.0001 acc=1.000
EfficientNet-B0 | epoch 07/30 train loss=0.0010 acc=1.000 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 08/30 train loss=0.0002 acc=1.000 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 09/30 train loss=0.0065 acc=0.999 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 10/30 train loss=0.0005 acc=1.000 val loss=0.0002 acc=1.000
EfficientNet-B0 | epoch 11/30 train loss=0.0011 acc=1.000 val loss=0.0000 acc=1.000
EfficientNet-B0 | epoch 12/30 train loss=0.0001 acc=1.000 val loss=0.0000 ac